In [1]:
import time
import csv
import os
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from llama_cpp import Llama

/home/prateek/Prateek/LaunchPad/week7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_MODEL_NAME  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH     = "/home/prateek/Prateek/LaunchPad/week8/Day4/adapters"
GGUF_PATH        = "/home/prateek/Prateek/LaunchPad/week8/Day4/quantized/model-INT4/tinyllama-q4.gguf"
RESULTS_PATH     = "/home/prateek/Prateek/LaunchPad/week8/Day4/benchmarks/results.csv"

In [4]:
PROMPTS = [
    "<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nWhat is compound interest?</s>\n<|assistant|>\n",
    "<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nA stock bought at $50 sold at $70 with $5 dividends. Calculate total return percentage.</s>\n<|assistant|>\n",
    "<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nExtract financial figures: Apple Q3 revenue $81.8B up 5% YoY, net income $19.9B.</s>\n<|assistant|>\n",
]

results = []

# ─────────────────────────────────────────
# HELPER — RAM USAGE
# ─────────────────────────────────────────
def get_ram_usage_gb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 ** 3), 2)

In [5]:
# ─────────────────────────────────────────
# 1. BASE MODEL (no fine-tuning)
# ─────────────────────────────────────────
print("\n" + "="*55)
print(" TEST 1: BASE MODEL (no fine-tuning)")
print("="*55)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

for prompt in PROMPTS:
    ram_before = get_ram_usage_gb()
    inputs = base_tokenizer(prompt, return_tensors="pt")
    
    start = time.time()
    with torch.no_grad():
        output = base_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
        )
    elapsed = time.time() - start
    
    input_len  = inputs["input_ids"].shape[1]
    output_len = output.shape[1] - input_len
    tok_per_sec = output_len / elapsed
    ram_after  = get_ram_usage_gb()
    response   = base_tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

    print(f"\nPrompt  : {prompt[50:90]}...")
    print(f"Response: {response[:100]}...")
    print(f"Tok/sec : {tok_per_sec:.2f} | Latency: {elapsed:.2f}s | RAM: {ram_after}GB")

    results.append({
        "model"        : "BASE",
        "prompt"       : prompt[30:70],
        "tokens_per_sec": round(tok_per_sec, 2),
        "latency_sec"  : round(elapsed, 2),
        "ram_gb"       : ram_after,
        "response"     : response[:150],
    })

del base_model  # free memory before loading next model



 TEST 1: BASE MODEL (no fine-tuning)


`torch_dtype` is deprecated! Use `dtype` instead!



Prompt  : |>
What is compound interest?</s>
<|assi...
Response: Compound interest is a type of interest rate that compounds over time. It is an interest rate that i...
Tok/sec : 6.84 | Latency: 14.62s | RAM: 4.62GB

Prompt  : |>
A stock bought at $50 sold at $70 wit...
Response: To calculate the total return percentage, we need to first find the compound annual growth rate (CAG...
Tok/sec : 6.40 | Latency: 15.62s | RAM: 4.63GB

Prompt  : |>
Extract financial figures: Apple Q3 r...
Response: Extract: Apple Q3 Revenue

Q3 2021 Revenue: $81.8 billion

YoY Growth: 5%

Net Income: $19.9 billion...
Tok/sec : 5.81 | Latency: 17.22s | RAM: 4.63GB
